<a href="https://colab.research.google.com/github/MananAslamDev/ML-Stuff/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method: Random Forest Classifier

Why: In Week 4, our baseline used a rigid, hardcoded rule (if impressions > 50 and sessions == 0, flag as an opportunity). However, traffic decay is rarely that strictly linear. A Random Forest can learn the complex, non-linear thresholds across multiple features (like impressions vs. AI search visibility) without overfitting. It will give us a probabilistic score rather than a binary "yes/no," allowing us to rank our content opportunities much more accurately than a simple rule.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split Design:
We will query our mid-panel month (March 2026) to avoid the non-random final month trap. We will use a standard 80/20 train-test split. The model will train on 80% of the URLs, and we will evaluate its performance on the remaining 20% to ensure it can generalize to unseen data.

In [3]:
import duckdb
import pandas as pd
from google.colab import userdata
from sklearn.model_selection import train_test_split

# 1. Authenticate with Hugging Face using the updated Secret syntax
hf_token = userdata.get('HF_TOKEN')
conn = duckdb.connect()
conn.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

# 2. Query the mid-panel month (March 2026)
query = """
    SELECT
        content_hash_id,
        gsc_impressions,
        ga4_sessions,
        ai_gemini,
        ai_copilot
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    WHERE gsc_impressions IS NOT NULL
"""
df = conn.execute(query).df()
df = df.fillna(0)

# 3. Define our target: Should this page have traffic? (Sessions > 0)
y = (df['ga4_sessions'] > 0).astype(int)

# 4. Define our features
X = df[['gsc_impressions', 'ai_gemini', 'ai_copilot']]

# 5. Split the data 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training rows: {len(X_train)} | Testing rows: {len(X_test)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Training rows: 7873102 | Testing rows: 1968276


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

# --- THE W04 BASELINE ---
# Rule: If a page gets more than 10 impressions, we expect it to get at least 1 session.
baseline_preds = (X_test['gsc_impressions'] > 10).astype(int)

# --- THE ML MODEL ---
# Train a Random Forest (max_depth=5 to prevent overfitting)
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)
model_preds = rf.predict(X_test)

# --- COMPARISON ---
print("--- PERFORMANCE COMPARISON ---")
print(f"Baseline Accuracy: {accuracy_score(y_test, baseline_preds):.3f}")
print(f"Model Accuracy:    {accuracy_score(y_test, model_preds):.3f}")
print("-" * 30)
print(f"Baseline Precision: {precision_score(y_test, baseline_preds, zero_division=0):.3f}")
print(f"Model Precision:    {precision_score(y_test, model_preds, zero_division=0):.3f}")

--- PERFORMANCE COMPARISON ---
Baseline Accuracy: 0.813
Model Accuracy:    0.959
------------------------------
Baseline Precision: 0.154
Model Precision:    0.991


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [5]:
# Look under the hood at what the model learned
importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("--- FEATURE IMPORTANCE ---")
print(importances)

--- FEATURE IMPORTANCE ---
           Feature  Importance
0  gsc_impressions    0.977127
1        ai_gemini    0.021359
2       ai_copilot    0.001514


Interpretation:
The ML model clearly outperformed the rigid baseline rule (especially in precision, meaning fewer false positives).
Looking at the feature importances, gsc_impressions is unsurprisingly the strongest predictor of whether a page should get traffic, but the model successfully learned how to weigh the AI visibility features (ai_gemini, ai_copilot) to refine its predictions.
The Errors: When the model is "wrong" (it predicts traffic, but actual sessions are 0), those aren't actually failures—those are our Content Refresh Targets. The math says they should be getting clicks based on their visibility profile, meaning the title or meta description is failing to convert the impression.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.